In [1]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

import torch

In [2]:
X, y = load_iris(return_X_y=True)

In [3]:
X.shape

(150, 4)

In [4]:
np.unique(y)

array([0, 1, 2])

In [5]:
class FCN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.fcn = torch.nn.Sequential(
            torch.nn.Linear(4, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 3)
        )

    def forward(self, X):
        X = self.fcn(X)
        return X

In [6]:
model = FCN()

In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [8]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.nn.functional.one_hot(torch.tensor(y)).to(dtype=torch.float32)

In [9]:
train_ds = torch.utils.data.TensorDataset(X_tensor, y_tensor)

In [10]:
mini_batch_size = 64

In [11]:
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False)

In [12]:
optimizer = torch.optim.Adam(model.parameters())

In [13]:
def fit(epochs, model, optimizer, train_dl):
    loss_func = torch.nn.CrossEntropyLoss()

    # loop over epochs
    for epoch in range(epochs):
        model.train()

        # loop over mini-batches
        for X_mb, y_mb in train_dl:
            y_hat = model(X_mb)

            loss = loss_func(y_hat, y_mb)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        model.eval()
        with torch.no_grad():
            train_loss = sum(loss_func(model(X_mb), y_mb) for X_mb, y_mb in train_dl)
        print('epoch {}, training loss {}'.format(epoch + 1, train_loss / len(train_dl)))

    return model

In [14]:
epochs = 50

In [15]:
fit(epochs, model, optimizer, train_dl)

epoch 1, training loss 1.10474693775177
epoch 2, training loss 1.0924259424209595
epoch 3, training loss 1.0843623876571655
epoch 4, training loss 1.0766922235488892
epoch 5, training loss 1.0693460702896118
epoch 6, training loss 1.0652521848678589
epoch 7, training loss 1.04791259765625
epoch 8, training loss 1.0321311950683594
epoch 9, training loss 1.0294092893600464
epoch 10, training loss 1.0130869150161743
epoch 11, training loss 0.9981827139854431
epoch 12, training loss 0.9828872084617615
epoch 13, training loss 0.9638188481330872
epoch 14, training loss 0.9363214373588562
epoch 15, training loss 0.9335185885429382
epoch 16, training loss 0.9141392111778259
epoch 17, training loss 0.9036857485771179
epoch 18, training loss 0.8459832072257996
epoch 19, training loss 0.8693700432777405
epoch 20, training loss 0.8300307393074036
epoch 21, training loss 0.8081138134002686
epoch 22, training loss 0.7704541683197021
epoch 23, training loss 0.7619848251342773
epoch 24, training loss 

FCN(
  (fcn): Sequential(
    (0): Linear(in_features=4, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=20, bias=True)
    (5): ReLU()
    (6): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [16]:
with torch.no_grad():
    yhat_train = model(train_ds[:][0])

In [17]:
yhat_train

tensor([[ 2.6317e+00, -1.4632e+00, -2.9861e+00],
        [ 2.0626e+00, -9.6529e-01, -3.0213e+00],
        [ 2.6338e+00, -1.4075e+00, -3.1457e+00],
        [ 2.5383e+00, -1.2971e+00, -3.2413e+00],
        [ 2.8089e+00, -1.6029e+00, -3.1198e+00],
        [ 2.7491e+00, -1.5904e+00, -2.9841e+00],
        [ 2.8688e+00, -1.6229e+00, -3.2332e+00],
        [ 2.5969e+00, -1.4101e+00, -3.0130e+00],
        [ 2.3907e+00, -1.1499e+00, -3.3759e+00],
        [ 2.2456e+00, -1.0828e+00, -3.1401e+00],
        [ 2.6160e+00, -1.4644e+00, -2.9309e+00],
        [ 2.7416e+00, -1.4976e+00, -3.1724e+00],
        [ 2.1762e+00, -1.0162e+00, -3.2013e+00],
        [ 2.7639e+00, -1.4275e+00, -3.5113e+00],
        [ 2.7405e+00, -1.5752e+00, -2.9850e+00],
        [ 3.1820e+00, -1.8951e+00, -3.3864e+00],
        [ 2.7902e+00, -1.6330e+00, -3.0193e+00],
        [ 2.5856e+00, -1.4556e+00, -2.8969e+00],
        [ 2.4636e+00, -1.3627e+00, -2.7604e+00],
        [ 2.9017e+00, -1.6881e+00, -3.1641e+00],
        [ 2.2277e+00

In [18]:
yhat_train = yhat_train.argmax(dim=1).numpy()

In [19]:
accuracy_score(y, yhat_train)

0.96